In [ ]:

def generate_story(idea, genre, style, length, model_name):
    if not idea or not idea.strip():
        return "⚠️ Please enter a story idea first."

    try:
        tokenizer, model = load_model(model_name)

        prompt = (
            f"Write an original {genre} story in a {style} style. "
            f"Story idea: {idea}. "
            f"Write a complete story with a beginning, middle, and ending. "
            f"Length: {length}. "
            f"Start the story immediately. Do not repeat these instructions."
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        import torch

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                min_new_tokens=40,
                num_beams=4,
                do_sample=False,
                repetition_penalty=1.2,
                early_stopping=True
            )

        story = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        ).strip()

        if not story:
            return "⚠️ No story was generated. Please try again."

        return story

    except Exception as e:
        return f"⚠️ Story generation failed: {str(e)}"


In [ ]:

# ============================================================
# 👻 STORYBLOOM AI — HORROR STORY GENERATOR
# Hugging Face Transformers + Gradio
# ============================================================

!pip install -q "transformers<5.0.0" gradio sentencepiece

import os
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------------------------------------------
# 1. AVAILABLE HUGGING FACE MODELS
# ------------------------------------------------------------

MODELS = {
    "🌸 FLAN-T5 Small — Faster": "google/flan-t5-small",
    "💜 FLAN-T5 Base — Better detail": "google/flan-t5-base",
}

model_cache = {}

# ------------------------------------------------------------
# 2. LOAD MODEL
# ------------------------------------------------------------

def load_model(model_name):
    if model_name not in model_cache:
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        model.eval()

        model_cache[model_name] = (tokenizer, model)

    return model_cache[model_name]


# ------------------------------------------------------------
# 3. HORROR STORY GENERATOR
# ------------------------------------------------------------

def generate_story(idea, model_choice, horror_type, writing_style,
                   story_length, custom_ending):

    if not idea or not idea.strip():
        return "👻 Please enter your horror story idea first!"

    model_name = MODELS.get(model_choice, "google/flan-t5-small")

    try:
        status = "Generating your nightmare..."
        tokenizer, model = load_model(model_name)

        length_settings = {
            "Short": 100,
            "Medium": 180,
            "Long": 280,
        }

        max_tokens = length_settings.get(story_length, 180)

        prompt = f"""
Write an original and complete {horror_type} horror story.

Story idea:
{idea.strip()}

Writing style: {writing_style}

Ending:
{custom_ending}

Instructions:
- Begin with an interesting opening.
- Introduce believable characters and a creepy setting.
- Build suspense gradually.
- Include frightening details and mysterious events.
- Make the story coherent and enjoyable.
- Write actual story paragraphs, not advice or instructions.
- Do not repeat the prompt.
- End the story in a memorable way.

Write the story now:
"""

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                num_beams=4,
                do_sample=False,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3,
                early_stopping=True
            )

        story = tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True
        ).strip()

        # Detect empty or unusable model output
        if not story or len(story.split()) < 8:
            return (
                "The model returned a very short response. "
                "Try a simpler story idea or switch to FLAN-T5 Base."
            )

        return story

    except Exception as error:
        return (
            "⚠️ Story generation failed.\n\n"
            f"Error details: {str(error)}\n\n"
            "Please check your internet connection and model settings."
        )


# ------------------------------------------------------------
# 4. DOWNLOAD STORY
# ------------------------------------------------------------

def save_story(story):
    if not story or not story.strip():
        raise gr.Error("Generate a story before downloading!")

    file_path = "/content/storybloom_horror_story.txt"

    with open(file_path, "w", encoding="utf-8") as file:
        file.write(story)

    return file_path


# ------------------------------------------------------------
# 5. BEAUTIFUL PASTEL UI
# ------------------------------------------------------------

CSS = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
    background: linear-gradient(135deg, #fcf5ff, #fff0f5, #f5f0ff);
    font-family: 'Trebuchet MS', sans-serif !important;
}
#main-title {
    text-align: center;
    padding: 22px;
    border-radius: 22px;
    background: linear-gradient(135deg, #ead8ff, #ffdbea);
    border: 1px solid #efc8ef;
    margin-bottom: 18px;
}
#main-title h1 {
    color: #693c91;
    font-size: 35px;
}
#main-title p {
    color: #754d83;
    font-size: 16px;
}
.panel {
    border: 1px solid #efd8f4 !important;
    border-radius: 18px !important;
    padding: 18px !important;
    background: rgba(255,255,255,0.84) !important;
}
#generate-btn {
    background: linear-gradient(90deg, #a56be0, #ed86b6) !important;
    color: white !important;
    border: none !important;
    border-radius: 14px !important;
    font-weight: bold !important;
    min-height: 48px;
}
#story-output textarea {
    font-size: 16px !important;
    line-height: 1.8 !important;
    color: #493653 !important;
}
footer {
    display: none !important;
}
"""

# ------------------------------------------------------------
# 6. BUILD APP
# ------------------------------------------------------------

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="purple",
        secondary_hue="pink",
        neutral_hue="slate"
    ),
    css=CSS,
    title="StoryBloom AI — Horror Generator"
) as app:

    gr.HTML("""
    <div id="main-title">
        <h1>👻 StoryBloom AI</h1>
        <p>✨ Enter the shadows. Create your own nightmare. ✨</p>
        <p>AI-powered horror stories, mysterious characters and chilling endings.</p>
    </div>
    """)

    with gr.Row():

        # LEFT PANEL
        with gr.Column(scale=1, elem_classes=["panel"]):

            gr.Markdown("## 🕯️ Create Your Nightmare")

            model_choice = gr.Dropdown(
                choices=list(MODELS.keys()),
                value="🌸 FLAN-T5 Small — Faster",
                label="🤖 Choose Your AI Model"
            )

            idea = gr.Textbox(
                label="📝 Describe Your Idea or Character",
                placeholder=(
                    "Example: A girl moves into an abandoned hostel. "
                    "Every night, someone whispers her name from a locked room..."
                ),
                lines=5
            )

            horror_type = gr.Dropdown(
                choices=[
                    "Supernatural",
                    "Psychological",
                    "Ghost",
                    "Haunted House",
                    "Paranormal Mystery",
                    "Gothic",
                    "Creature Horror",
                    "Suspense Thriller"
                ],
                value="Supernatural",
                label="👁️ Horror Genre"
            )

            writing_style = gr.Dropdown(
                choices=[
                    "Dark and creepy",
                    "Suspenseful and mysterious",
                    "Cinematic and dramatic",
                    "Gothic and atmospheric",
                    "Fast-paced and terrifying"
                ],
                value="Dark and creepy",
                label="🦇 Writing Style"
            )

            story_length = gr.Radio(
                choices=["Short", "Medium", "Long"],
                value="Medium",
                label="📖 Story Length"
            )

            custom_ending = gr.Dropdown(
                choices=[
                    "A shocking twist",
                    "A terrifying ending",
                    "A mysterious cliffhanger",
                    "An unexpected revelation",
                    "A dark but satisfying ending"
                ],
                value="A shocking twist",
                label="🔮 Choose Your Ending"
            )

            generate_btn = gr.Button(
                "👻 Generate My Horror Story",
                variant="primary",
                elem_id="generate-btn"
            )

            clear_btn = gr.ClearButton(
                components=[
                    idea, horror_type, writing_style,
                    story_length, custom_ending
                ],
                value="🧹 Clear Inputs"
            )

        # RIGHT PANEL
        with gr.Column(scale=1, elem_classes=["panel"]):

            gr.Markdown("## 📖 Your Magical Story")

            gr.Markdown(
                "🌸 Your generated horror story will appear below."
            )

            story_output = gr.Textbox(
                label="✨ Generated Story",
                placeholder=(
                    "Your nightmare will appear here...\n\n"
                    "Enter an idea and click Generate My Horror Story."
                ),
                lines=20,
                max_lines=30,
                elem_id="story-output",
                interactive=False
            )

            with gr.Row():
                copy_btn = gr.Button("📋 Select Story to Copy")
                download_btn = gr.Button("📥 Download Story")

            download_file = gr.File(
                label="Your Story File",
                visible=False
            )

            gr.Markdown(
                "*✨ Tip: For a more detailed story, choose Medium or Long.*"
            )

    # Connect generation to the actual output textbox
    generate_btn.click(
        fn=generate_story,
        inputs=[
            idea,
            model_choice,
            horror_type,
            writing_style,
            story_length,
            custom_ending
        ],
        outputs=story_output
    )

    # Download generated story
    download_btn.click(
        fn=save_story,
        inputs=story_output,
        outputs=download_file
    )

    # Copy button: select the text so it can be copied
    copy_btn.click(
        fn=None,
        inputs=None,
        outputs=None,
        js="""
        () => {
            const box = document.querySelector(
                '#story-output textarea'
            );
            if (box) {
                box.focus();
                box.select();
                document.execCommand('copy');
            }
        }
        """
    )

    gr.Markdown("""
    <center>
    🌙 Made with imagination and AI · StoryBloom AI 💜
    </center>
    """)

# ------------------------------------------------------------
# 7. LAUNCH APP
# ------------------------------------------------------------

app.queue().launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5f24e2a0342f9ee782.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]